In [1]:
from pyspark.sql import functions as F

# ============================================================
# AirOps 360 - W5-09
# Gold dimensions + KPI definitions
# ============================================================

FACT_TABLE = "fact_flight_performance"
DAILY_AGG_TABLE = "agg_daily_origin_airport_performance"

DIM_DATE = "dim_date"
DIM_AIRPORT = "dim_airport"
DIM_CARRIER = "dim_carrier"
KPI_TABLE = "gold_kpi_snapshot"

EXPECTED_FACT_ROWS = 597_919

fact = spark.table(FACT_TABLE)
daily_agg = spark.table(DAILY_AGG_TABLE)

fact_rows = fact.count()
fact_distinct_keys = fact.select("flight_key").distinct().count()

assert fact_rows == EXPECTED_FACT_ROWS
assert fact_distinct_keys == EXPECTED_FACT_ROWS

print("GOLD INPUT READY")
print(f"Fact rows:             {fact_rows:,}")
print(f"Distinct flight_key:   {fact_distinct_keys:,}")

StatementMeta(, 549c6547-f5c3-405a-8c1d-6276214c7179, 3, Finished, Available, Finished, False)

GOLD INPUT READY
Fact rows:             597,919
Distinct flight_key:   597,919


In [2]:
# ============================================================
# DIM_DATE
#
# Grain:
#   1 row = 1 calendar date represented in the Gold fact
# ============================================================

dim_date = (
    fact
    .select(
        "date_key",
        "flight_date"
    )
    .distinct()

    .withColumn(
        "year",
        F.year("flight_date")
    )

    .withColumn(
        "quarter",
        F.quarter("flight_date")
    )

    .withColumn(
        "month",
        F.month("flight_date")
    )

    .withColumn(
        "day_of_month",
        F.dayofmonth("flight_date")
    )

    .withColumn(
        "day_of_week",
        F.dayofweek("flight_date")
    )

    .withColumn(
        "day_name",
        F.date_format("flight_date", "EEEE")
    )

    .orderBy("flight_date")
)


(
    dim_date.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(DIM_DATE)
)

print(f"dim_date rows: {dim_date.count():,}")

display(dim_date)

StatementMeta(, 549c6547-f5c3-405a-8c1d-6276214c7179, 4, Finished, Available, Finished, False)

dim_date rows: 30


SynapseWidget(Synapse.DataFrame, f9a0f97d-6cd4-4bd2-9950-6b104c9232e8)

In [3]:
# ============================================================
# DIM_CARRIER
#
# Grain:
#   1 row = 1 Reporting_Airline carrier code
# ============================================================

dim_carrier = (
    fact
    .select(
        "carrier_key",
        "carrier_code"
    )
    .distinct()
)

carrier_code_collisions = (
    dim_carrier
    .groupBy("carrier_code")
    .agg(
        F.countDistinct("carrier_key").alias("key_count")
    )
    .filter(F.col("key_count") > 1)
    .count()
)

carrier_key_collisions = (
    dim_carrier
    .groupBy("carrier_key")
    .agg(
        F.countDistinct("carrier_code").alias("code_count")
    )
    .filter(F.col("code_count") > 1)
    .count()
)

assert carrier_code_collisions == 0
assert carrier_key_collisions == 0


(
    dim_carrier.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(DIM_CARRIER)
)

print(f"dim_carrier rows:         {dim_carrier.count():,}")
print(f"Carrier code collisions:  {carrier_code_collisions}")
print(f"Carrier key collisions:   {carrier_key_collisions}")

display(
    dim_carrier
    .orderBy("carrier_code")
)

StatementMeta(, 549c6547-f5c3-405a-8c1d-6276214c7179, 5, Finished, Available, Finished, False)

dim_carrier rows:         13
Carrier code collisions:  0
Carrier key collisions:   0


SynapseWidget(Synapse.DataFrame, c30c57a8-fb91-4488-8cfe-5418a23f0f56)

In [4]:
# ============================================================
# DIM_AIRPORT
#
# Grain:
#   1 row = 1 distinct BTS airport code represented
#           by accepted Gold flights
#
# Same physical dimension supports:
#   origin_airport_key
#   destination_airport_key
# ============================================================

origin_airports = (
    fact
    .select(
        F.col("origin_airport_key")
         .alias("airport_key"),

        F.col("origin_airport_code")
         .alias("airport_code")
    )
)

destination_airports = (
    fact
    .select(
        F.col("destination_airport_key")
         .alias("airport_key"),

        F.col("destination_airport_code")
         .alias("airport_code")
    )
)

dim_airport = (
    origin_airports
    .unionByName(destination_airports)
    .distinct()
)


# ------------------------------------------------------------
# Collision checks
# ------------------------------------------------------------

airport_code_collisions = (
    dim_airport
    .groupBy("airport_code")
    .agg(
        F.countDistinct("airport_key").alias("key_count")
    )
    .filter(F.col("key_count") > 1)
    .count()
)

airport_key_collisions = (
    dim_airport
    .groupBy("airport_key")
    .agg(
        F.countDistinct("airport_code").alias("code_count")
    )
    .filter(F.col("code_count") > 1)
    .count()
)

assert airport_code_collisions == 0
assert airport_key_collisions == 0


(
    dim_airport.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(DIM_AIRPORT)
)


print(f"dim_airport rows:        {dim_airport.count():,}")
print(f"Airport code collisions: {airport_code_collisions}")
print(f"Airport key collisions:  {airport_key_collisions}")

display(
    dim_airport
    .orderBy("airport_code")
)

StatementMeta(, 549c6547-f5c3-405a-8c1d-6276214c7179, 6, Finished, Available, Finished, False)

dim_airport rows:        342
Airport code collisions: 0
Airport key collisions:  0


SynapseWidget(Synapse.DataFrame, c790fdc3-084e-4e6f-9968-fc3af81f7274)

In [5]:
# ============================================================
# DIMENSION RELATIONSHIP COVERAGE
# ============================================================

date_dim = spark.table(DIM_DATE)
airport_dim = spark.table(DIM_AIRPORT)
carrier_dim = spark.table(DIM_CARRIER)


# ------------------------------------------------------------
# Dimension grain uniqueness
# ------------------------------------------------------------

date_rows = date_dim.count()
date_distinct_keys = date_dim.select("date_key").distinct().count()

airport_rows = airport_dim.count()
airport_distinct_keys = airport_dim.select("airport_key").distinct().count()

carrier_rows = carrier_dim.count()
carrier_distinct_keys = carrier_dim.select("carrier_key").distinct().count()

assert date_rows == date_distinct_keys
assert airport_rows == airport_distinct_keys
assert carrier_rows == carrier_distinct_keys


# ------------------------------------------------------------
# Fact -> dimension coverage
# LEFT ANTI returns fact keys that cannot find a dimension match
# ------------------------------------------------------------

missing_dates = (
    fact
    .select("date_key")
    .distinct()
    .join(
        date_dim.select("date_key"),
        on="date_key",
        how="left_anti"
    )
    .count()
)

missing_origin_airports = (
    fact
    .select(
        F.col("origin_airport_key")
         .alias("airport_key")
    )
    .distinct()
    .join(
        airport_dim.select("airport_key"),
        on="airport_key",
        how="left_anti"
    )
    .count()
)

missing_destination_airports = (
    fact
    .select(
        F.col("destination_airport_key")
         .alias("airport_key")
    )
    .distinct()
    .join(
        airport_dim.select("airport_key"),
        on="airport_key",
        how="left_anti"
    )
    .count()
)

missing_carriers = (
    fact
    .select("carrier_key")
    .distinct()
    .join(
        carrier_dim.select("carrier_key"),
        on="carrier_key",
        how="left_anti"
    )
    .count()
)


assert missing_dates == 0
assert missing_origin_airports == 0
assert missing_destination_airports == 0
assert missing_carriers == 0


print("=" * 68)
print("DIMENSION COVERAGE")
print("=" * 68)

print(
    f"Date rows / distinct keys:        "
    f"{date_rows:,} / {date_distinct_keys:,}"
)

print(
    f"Airport rows / distinct keys:     "
    f"{airport_rows:,} / {airport_distinct_keys:,}"
)

print(
    f"Carrier rows / distinct keys:     "
    f"{carrier_rows:,} / {carrier_distinct_keys:,}"
)

print()

print(f"Missing date keys:                {missing_dates}")
print(f"Missing origin airport keys:      {missing_origin_airports}")
print(f"Missing destination airport keys: {missing_destination_airports}")
print(f"Missing carrier keys:             {missing_carriers}")

print()
print("DIMENSION COVERAGE STATUS: PASS")

StatementMeta(, 549c6547-f5c3-405a-8c1d-6276214c7179, 7, Finished, Available, Finished, False)

DIMENSION COVERAGE
Date rows / distinct keys:        30 / 30
Airport rows / distinct keys:     342 / 342
Carrier rows / distinct keys:     13 / 13

Missing date keys:                0
Missing origin airport keys:      0
Missing destination airport keys: 0
Missing carrier keys:             0

DIMENSION COVERAGE STATUS: PASS


In [6]:
# ============================================================
# KPI DEFINITIONS
#
# KPI 1: Total Flights
#   numerator/measure = all fact rows
#
# KPI 2: Cancellation Rate
#   numerator   = cancelled flights
#   denominator = all scheduled flights
#   NULL rule   = cancelled_flag must NOT be NULL
#
# KPI 3: Arrival Delay 15 Rate
#   numerator   = flights where arrival_delayed_15_flag = True
#   denominator = flights where arrival_delayed_15_flag IS NOT NULL
#   NULL rule   = NULL excluded from denominator; it is NOT "on time"
# ============================================================


null_cancelled_flags = (
    fact
    .filter(
        F.col("cancelled_flag").isNull()
    )
    .count()
)

assert null_cancelled_flags == 0, (
    f"STOP: found {null_cancelled_flags:,} NULL cancelled flags"
)


kpi = (
    fact

    .agg(
        # KPI 1
        F.count("*")
         .alias("total_flights"),

        # KPI 2 numerator
        F.sum(
            F.when(
                F.col("cancelled_flag") == True,
                1
            ).otherwise(0)
        ).alias("cancelled_flights"),

        # KPI 3 denominator
        F.sum(
            F.when(
                F.col("arrival_delayed_15_flag").isNotNull(),
                1
            ).otherwise(0)
        ).alias("arrival_delay_eligible_flights"),

        # KPI 3 numerator
        F.sum(
            F.when(
                F.col("arrival_delayed_15_flag") == True,
                1
            ).otherwise(0)
        ).alias("arrival_delayed_15_flights")
    )

    # Cancellation denominator is explicitly all scheduled flights
    .withColumn(
        "cancellation_denominator",
        F.col("total_flights")
    )

    .withColumn(
        "cancellation_rate",
        F.when(
            F.col("cancellation_denominator") > 0,

            F.col("cancelled_flights").cast("double")
            /
            F.col("cancellation_denominator").cast("double")
        )
        .otherwise(
            F.lit(None).cast("double")
        )
    )

    # Transparency: how many rows are intentionally excluded
    # from arrival-delay denominator?
    .withColumn(
        "arrival_delay_unknown_flights",
        F.col("total_flights")
        -
        F.col("arrival_delay_eligible_flights")
    )

    .withColumn(
        "arrival_delay_15_rate",
        F.when(
            F.col("arrival_delay_eligible_flights") > 0,

            F.col("arrival_delayed_15_flights").cast("double")
            /
            F.col("arrival_delay_eligible_flights").cast("double")
        )
        .otherwise(
            F.lit(None).cast("double")
        )
    )

    .withColumn(
        "kpi_scope",
        F.lit("ALL_CURRENT_GOLD_FLIGHTS")
    )

    .withColumn(
        "_gold_version",
        F.lit("gold_mvp_v0.1")
    )

    .withColumn(
        "_gold_published_at_utc",
        F.current_timestamp()
    )
)


(
    kpi.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(KPI_TABLE)
)


display(
    spark.table(KPI_TABLE)
)

StatementMeta(, 549c6547-f5c3-405a-8c1d-6276214c7179, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d908cff-7671-4ff3-b3bd-394c2dae8311)

In [7]:
# ============================================================
# KPI RECONCILIATION
# ============================================================

kpi_row = spark.table(KPI_TABLE).first()


daily_totals = (
    daily_agg
    .agg(
        F.sum("total_flights")
         .alias("total_flights"),

        F.sum("cancelled_flights")
         .alias("cancelled_flights"),

        F.sum("arrival_delay_eligible_flights")
         .alias("arrival_delay_eligible_flights"),

        F.sum("arrival_delayed_15_flights")
         .alias("arrival_delayed_15_flights")
    )
    .first()
)


# ------------------------------------------------------------
# Independent total reconciliation
# ------------------------------------------------------------

assert kpi_row["total_flights"] == fact_rows

assert (
    kpi_row["total_flights"]
    ==
    daily_totals["total_flights"]
)

assert (
    kpi_row["cancelled_flights"]
    ==
    daily_totals["cancelled_flights"]
)

assert (
    kpi_row["arrival_delay_eligible_flights"]
    ==
    daily_totals["arrival_delay_eligible_flights"]
)

assert (
    kpi_row["arrival_delayed_15_flights"]
    ==
    daily_totals["arrival_delayed_15_flights"]
)


# ------------------------------------------------------------
# Logical denominator checks
# ------------------------------------------------------------

assert (
    kpi_row["cancelled_flights"]
    <=
    kpi_row["cancellation_denominator"]
)

assert (
    kpi_row["arrival_delayed_15_flights"]
    <=
    kpi_row["arrival_delay_eligible_flights"]
)

assert (
    kpi_row["arrival_delay_eligible_flights"]
    <=
    kpi_row["total_flights"]
)

assert (
    kpi_row["arrival_delay_unknown_flights"]
    ==
    (
        kpi_row["total_flights"]
        -
        kpi_row["arrival_delay_eligible_flights"]
    )
)


print("=" * 72)
print("AIROPS 360 - W5-09 KPI EVIDENCE")
print("=" * 72)

print(f"Total flights:                     {kpi_row['total_flights']:,}")

print()
print("CANCELLATION KPI")
print("-" * 72)
print(f"Cancelled flights:                 {kpi_row['cancelled_flights']:,}")
print(f"Denominator - scheduled flights:   {kpi_row['cancellation_denominator']:,}")
print(f"Cancellation rate:                 {kpi_row['cancellation_rate']:.4%}")
print(f"NULL cancellation flags:           {null_cancelled_flags:,}")

print()
print("ARRIVAL DELAY >= 15 MIN KPI")
print("-" * 72)
print(f"Delayed >=15 min flights:          {kpi_row['arrival_delayed_15_flights']:,}")
print(f"Eligible flights:                  {kpi_row['arrival_delay_eligible_flights']:,}")
print(f"Excluded NULL outcomes:            {kpi_row['arrival_delay_unknown_flights']:,}")
print(f"Arrival delay 15 rate:             {kpi_row['arrival_delay_15_rate']:.4%}")

print()
print("DAILY AGGREGATE RECONCILIATION")
print("-" * 72)
print(f"Daily total flights:               {daily_totals['total_flights']:,}")
print(f"Daily cancelled flights:           {daily_totals['cancelled_flights']:,}")
print(f"Daily eligible arrival outcomes:   {daily_totals['arrival_delay_eligible_flights']:,}")
print(f"Daily delayed >=15 min flights:    {daily_totals['arrival_delayed_15_flights']:,}")

print()
print("KPI RECONCILIATION STATUS: PASS")
print("=" * 72)

StatementMeta(, 549c6547-f5c3-405a-8c1d-6276214c7179, 9, Finished, Available, Finished, False)

AIROPS 360 - W5-09 KPI EVIDENCE
Total flights:                     597,919

CANCELLATION KPI
------------------------------------------------------------------------
Cancelled flights:                 5,402
Denominator - scheduled flights:   597,919
Cancellation rate:                 0.9035%
NULL cancellation flags:           0

ARRIVAL DELAY >= 15 MIN KPI
------------------------------------------------------------------------
Delayed >=15 min flights:          119,417
Eligible flights:                  591,206
Excluded NULL outcomes:            6,713
Arrival delay 15 rate:             20.1989%

DAILY AGGREGATE RECONCILIATION
------------------------------------------------------------------------
Daily total flights:               597,919
Daily cancelled flights:           5,402
Daily eligible arrival outcomes:   591,206
Daily delayed >=15 min flights:    119,417

KPI RECONCILIATION STATUS: PASS


In [8]:
# ============================================================
# W5-09 AFFECTED-CHANGE RELEASE GATE
# ============================================================

checks = [
    ("fact_grain_unique",
     fact_rows == fact_distinct_keys),

    ("date_dimension_unique",
     date_rows == date_distinct_keys),

    ("airport_dimension_unique",
     airport_rows == airport_distinct_keys),

    ("carrier_dimension_unique",
     carrier_rows == carrier_distinct_keys),

    ("date_join_coverage",
     missing_dates == 0),

    ("origin_airport_join_coverage",
     missing_origin_airports == 0),

    ("destination_airport_join_coverage",
     missing_destination_airports == 0),

    ("carrier_join_coverage",
     missing_carriers == 0),

    ("total_flights_reconciled",
     kpi_row["total_flights"]
     == daily_totals["total_flights"]),

    ("cancelled_flights_reconciled",
     kpi_row["cancelled_flights"]
     == daily_totals["cancelled_flights"]),

    ("arrival_eligible_reconciled",
     kpi_row["arrival_delay_eligible_flights"]
     == daily_totals["arrival_delay_eligible_flights"]),

    ("arrival_delayed_reconciled",
     kpi_row["arrival_delayed_15_flights"]
     == daily_totals["arrival_delayed_15_flights"]),
]


failed = [
    name
    for name, passed in checks
    if not passed
]


print("=" * 72)
print("AIROPS 360 - W5-09 RELEASE GATE")
print("=" * 72)

for name, passed in checks:
    print(
        f"{'PASS' if passed else 'FAIL':<5} | {name}"
    )

print("-" * 72)
print(f"Passed: {len(checks) - len(failed)} / {len(checks)}")


assert not failed, (
    f"W5-09 release gate failed: {failed}"
)

print("W5-09 GOLD RELEASE GATE: PASS")

StatementMeta(, 549c6547-f5c3-405a-8c1d-6276214c7179, 10, Finished, Available, Finished, False)

AIROPS 360 - W5-09 RELEASE GATE
PASS  | fact_grain_unique
PASS  | date_dimension_unique
PASS  | airport_dimension_unique
PASS  | carrier_dimension_unique
PASS  | date_join_coverage
PASS  | origin_airport_join_coverage
PASS  | destination_airport_join_coverage
PASS  | carrier_join_coverage
PASS  | total_flights_reconciled
PASS  | cancelled_flights_reconciled
PASS  | arrival_eligible_reconciled
PASS  | arrival_delayed_reconciled
------------------------------------------------------------------------
Passed: 12 / 12
W5-09 GOLD RELEASE GATE: PASS
